# DPO orca-math-korean

**Dataset:**
- https://huggingface.co/datasets/microsoft/orca-math-word-problems-200k
- https://huggingface.co/datasets/kuotient/orca-math-korean-dpo-pairs

**Model:**
- https://huggingface.co/soka0000/vclm-korean-7b

In [ ]:
!pip install -Uqq datasets transformers hf_transfer accelerate peft trl wandb scikit-learn


Usage:   
  pip <command> [options]

no such option: -U


In [2]:
from datasets import load_dataset

dataset = load_dataset('kuotient/orca-math-korean-dpo-pairs', split='train')
SAMPLE_SIZE = 10000
dataset = dataset.select(range(SAMPLE_SIZE))
print(len(dataset))
print(dataset[100])

10000
{'system': '당신은 인공지능 어시스턴트입니다.', 'question': '한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?', 'chosen': '먼저 모든 측정값을 동일한 단위로 변환하여 계산을 쉽게 해봅시다. 모든 것을 밀리리터(ml)로 변환하겠습니다.\n\n배럴에는 12리터와 400밀리리터가 들어 있습니다. 1리터는 1000밀리리터와 같으므로 12리터를 밀리리터로 변환할 수 있습니다:\n12리터 = 12 * 1000밀리리터 = 12000밀리리터\n\n이제 이미 밀리리터 단위로 표시된 400밀리리터를 더합니다:\n12000밀리리터 + 400밀리리터 = 12400밀리리터\n\n따라서 배럴에는 총 12400밀리리터의 석유 가 들어 있습니다.\n\nB 배럴에는 7600밀리리터의 석유가 들어 있습니다.\n\n두 배럴의 석유 양을 동일하게 하려면 두 양의 평균을 구해야 합니다:\n두 배럴의 총량 = 12400밀리리터(A배럴) + 7600밀리리터(B배럴)\n두 배럴의 총량 = 20000밀리리터\n\n이제 이 총량을 2로 나누어 각 배럴에 해당하는 양을 구합니다:\n각 배럴의 동일한 양 = 20000밀리리터 / 2\n각 배럴의 동일한 양 = 10000밀리리터\n\n현재 A 배럴에는 12400밀리리터가 있으므로, 두 배럴의 양이 각각 10000밀리리터가 되도록 일부를 B 배럴로 이동해야 합니다.\n\nA에서 B로 이동할 양 = 12400밀리리터(A 배럴) - 10000밀리리터(동일한 양)\nA에서 B로 이동할 양 = 2400밀리리터\n\n따라서 두 배럴의 석유 양을 동일하게 하기 위해 A 배럴에서 B 배럴로 2400밀리리터(또는 2.4리터)를 이동해야 합니다.', 'rejected': ' 먼저 B 배럴의 석유 양을 리터로 변환하여 A 배럴의 석유 양과 비교해야 합니다.\n\nB 배럴의

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'soka0000/vclm-korean-7b'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--soka0000--vclm-korean-7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] You are using a model of type `Soka1.0` to instantiate a model of type ``. This may be expecte

In [6]:
def preprocess_dpo_data(example):
    messages = [
        {'role': 'system', 'content': example['system']},
        {'role': 'user', 'content': example['question']},
    ]

    prompt_style = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return {
        'prompt': prompt_style,
        'chosen': example['chosen'],
        'rejected': example['rejected']
    }

dataset_preprocessed = dataset.map(preprocess_dpo_data)

print(f'변경 전 컬럼: {dataset.column_names}')
print(f'변경  컬럼: {dataset_preprocessed.column_names}')

Map: 100%|██████████| 10000/10000 [00:01<00:00, 9881.02 examples/s]

변경 전 컬럼: ['system', 'question', 'chosen', 'rejected']
변경  컬럼: ['system', 'question', 'chosen', 'rejected', 'prompt']


In [7]:
dataset_preprocessed['prompt'][100]

'<|im_start|>system\n당신은 인공지능 어시스턴트입니다.<|im_end|>\n<|im_start|>user\n한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?<|im_end|>\n<|im_start|>assistant\n'

In [8]:
train_size = int(len(dataset_preprocessed) * 0.8)
val_size = int(len(dataset_preprocessed) * 0.1)
test_size = int(len(dataset_preprocessed) * 0.1)

train_dataset = dataset_preprocessed.select(range(train_size))
val_dataset = dataset_preprocessed.select(range(train_size, train_size + val_size))
test_dataset = dataset_preprocessed.select(range(train_size + val_size, len(dataset_preprocessed)))

print(len(train_dataset), len(val_dataset), len(test_dataset))

8000 1000 1000


In [10]:
def generate_response(model, tokenizer, question):
    prompt = question
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=1024,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.5,
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
        generated_text = tokenizer.decode(outputs[0])
        return generated_text(prompt, '').strip()

In [11]:
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']

    print(f'질문: {question} / 정답: {answer}')

    generated_answer = generate_response(model, tokenizer, question)
    print(f'모델 생성 답변: {generated_answer}')
    print('=' * 100)

질문: <|im_start|>system
당신은 인공지능 어시스턴트입니다.<|im_end|>
<|im_start|>user
알리사와 아비게일은 과학 프로젝트를 위해 빈 캔 100개를 모아야 합니다. 오늘 현재 알리사는 빈 캔을 몇 개 모았고, 아비게일은 빈 캔 43개를 모았습니다. 그들은 빈 캔을 27개 더 모아야 합니다. 지금까지 알리사가 모은 빈 캔은 몇 개인가요?<|im_end|>
<|im_start|>assistant
 / 정답: 알리사가 얼마나 많은 빈 캔을 모았는지 알아내려면, 아비게일이 모은 캔의 수와 프로젝트에 필요한 총 캔 수에서 아직 모아야 할 캔의 수를 빼야 합니다.

알리사의 캔 + 아비게일의 캔 + 아직 필요한 캔 = 필요한 총 캔 수입니다.
알리사의 캔 = 필요한 총 캔 수 - (아비게일의 캔 + 아직 필요한 캔)

우리는 이것을 알고 있습니다:
필요한 총 캔 수 = 100
아비게일의 캔 = 43
아직 필요한 캔 = 27

이제 숫자를 연결할 수 있습니다:

알리사의 캔 = 100 - (43 + 27)
알리사의 캔 = 100 - 70
알리사의 캔 = 30

알리사는 지금까지 빈 캔 30개를 모았습니다.


NameError: name 'model' is not defined

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
reference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)

reference_model.eval()
for param in reference_model.parameters():
    param.requires_grad = False

In [ ]:
import wandb

wandb.login()

In [ ]:
from trl import DPOTrainer, DPOConfig

hub_model_id = 'kty2001/vclm-korean-7b-orca-math-korean-dpo'

training_args = DPOConfig(  # DPO 학습 하이퍼파라미터/로깅/저장 설정
    output_dir='vclm-korean-7b-orca-math-korean-dpo',  # 체크포인트/로그 저장 폴더
    num_train_epochs=1,  # 전체 데이터를 1번 반복 학습
    per_device_train_batch_size=2,  # GPU 1개당 배치 크기
    gradient_accumulation_steps=4,  # 4번 누적 후 업데이트(실제 배치 효과: 2*4=8)
    learning_rate=5e-5,  # 학습률
    eval_strategy="steps",  # 일정 step마다 평가 수행
    save_strategy="steps",  # 일정 step마다 저장 수행
    logging_steps=50,  # 50 step마다 학습 로그 출력
    fp16=False,  # fp16 비활성화(여기서는 bf16 사용)
    bf16=True,  # bfloat16 사용(지원 GPU에서 안정적/빠름)
    tf32=True,  # Ampere 이상에서 matmul 가속 옵션(정밀도 약간 완화)
    beta=0.1,  # DPO의 beta(선호 강도 조절)
    max_length=512,  # prompt+답변을 포함한 최대 길이
    remove_unused_columns=False,  # DPO에 필요한 컬럼이 제거되지 않도록 유지
    push_to_hub=True,  # 학습 결과를 Hugging Face Hub로 업로드
    hub_model_id=hub_model_id,  # 업로드할 저장소 이름
    hub_strategy="end",  # 학습 끝난 뒤 한 번만 업로드
    report_to=['wandb']  # wandb로 학습 로그 전송
)

dpo_trainer = DPOTrainer(  # DPO Trainer 생성(정책모델 vs 참조모델 비교 학습)
    model=model,  # 정책모델(LoRA 적용된 학습 대상)
    ref_model=reference_model,  # 참조모델(고정, 비교 기준)
    args=training_args,  # 위에서 만든 학습 설정
    train_dataset=train_dataset,  # 학습 데이터(prompt/chosen/rejected)
    eval_dataset=val_dataset,  # 검증 데이터
    processing_class=tokenizer  # 토큰화 처리(TRL 버전에 따라 tokenizer 인자명 상이 가능)
)

dpo_trainer.train()  # DPO 학습 시작

In [ ]:
# dpo_trainer.push_to_hub('Commit')

In [ ]:
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']

    print(f'질문: {question} / 정답: {answer}')

    generated_answer = generate_response(model, tokenizer, question)
    print(f'모델 생성 답변: {generated_answer}')
    print('=' * 100)

In [1]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

def calculate_log_prob(model, tokenizer, prompt, response):
    """
    주어진 prompt에 대한 response의 log probability 평균 값 계산

    Args:
        model: HuggingFace AutoModelForCausalLM (or similar)
        tokenizer: HuggingFace AutoTokenizer
        prompt (str): prompt text
        response (str): response text

    Returns:
        float: mean log probability of response tokens
    """
    full_text = prompt + " " + response
    inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    prompt_tokens = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    prompt_len = prompt_tokens['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = input_ids[..., 1:].contiguous()

    log_probs = F.log_softmax(shift_logits, dim=-1)

    true_log_probs = torch.gather(
        log_probs,
        2,
        shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    seq_len = shift_labels.shape[1]
    if prompt_len >= seq_len:
        valid_log_probs = true_log_probs[:, -1:]
    else:
        start_idx = max(0, prompt_len-1)
        valid_log_probs = true_log_probs[:, start_idx:]

    avg_log_prob = valid_log_probs.mean().item()
    return avg_log_prob

# chosen_prob = get_logprob(model, tokenizer, prompt, chosen)
# rejected_prob = get_logprob(model, tokenizer, prompt, rejected)
# print(f'선호 답변 생성 확률: {chosen_prob}, 비선호 답변 생성 확률: {rejected_prob}')

In [2]:
def calculate_preference_accuracy(model, tokenizer, dataset, num_samples=100):
    """
    모델이 선호 변에 비선호 답변보다 더 높은 확률 부여하는지 평가
    """
    correct = 0
    total = min(num_samples, len(dataset))

    print(f'전체 개수: {total}')

    model.eval()

    for idx in range(total):
        example = dataset[idx]
        prompt = example['prompt']
        chosen = example['chosen']
        rejected = example['rejected']

        chosen_score = calculate_log_prob(model, tokenizer, prompt, chosen)
        rejected_score = calculate_log_prob(model, tokenizer, prompt, rejected)

        if chosen_score > rejected_score:
            correct += 1

        if (idx + 1) % 10 == 0:
            print(f'{idx+1}/{total} Acc: {correct/(idx+1)*100:.2f}')

    accuracy = correct / tokenizer
    return accuracy

test_accuracy = calculate_preference_accuracy(model, tokenizer, test_dataset, num_samples=300)
print(test_accuracy)

NameError: name 'model' is not defined